In [ ]:
import os
import re
import hashlib
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.llms import Ollama
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings
from langchain.chains import create_retrieval_chain

class RAGChatbot:
    def __init__(self):
        self.retrieval_chain = None
        self.current_pdf_path = None

    def get_document_signature(self, file_path):
        """Generates a hash of the PDF file's content."""
        hasher = hashlib.sha256()
        try:
            with open(file_path, 'rb') as f:
                while chunk := f.read(4096):
                    hasher.update(chunk)
            return hasher.hexdigest()
        except Exception:
            return None

    def initialize_retrieval_chain(self, pdf_path, force_reprocess=False):
        """
        Initializes the Retrieval-Augmented Generation (RAG) chain.

        Args:
            pdf_path (str): Path to the PDF knowledge base.
            force_reprocess (bool): If True, forces re-processing the document
                                    even if a saved signature exists.
        
        Returns:
            dict: A dictionary with the status and a message.
        """
        self.current_pdf_path = pdf_path
        persist_directory = "./chroma_db"
        os.makedirs(persist_directory, exist_ok=True)
        signature_file = os.path.join(persist_directory, "doc_signature.txt")
        current_signature = self.get_document_signature(self.current_pdf_path)

        load_from_disk = False
        if not force_reprocess and os.path.exists(signature_file):
            with open(signature_file, 'r') as f:
                saved_signature = f.read()
            if saved_signature == current_signature:
                load_from_disk = True

        if load_from_disk:
            print("Loading existing embeddings...")
            embeddings = OllamaEmbeddings(model="nomic-embed-text")
            vector_store = Chroma(persist_directory=persist_directory, embedding_function=embeddings)
        else:
            print("Creating new embeddings...")
            try:
                loader = PyPDFLoader(self.current_pdf_path)
                docs = loader.load()
                text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
                all_splits = text_splitter.split_documents(docs)
                embeddings = OllamaEmbeddings(model="nomic-embed-text")
                vector_store = Chroma.from_documents(documents=all_splits, embedding=embeddings, persist_directory=persist_directory)
                with open(signature_file, 'w') as f:
                    f.write(current_signature)
            except Exception as e:
                return {"status": "error", "message": f"Failed to process PDF: {e}"}

        llm = Ollama(model="gemma3:1b")
        retriever = vector_store.as_retriever()
        prompt = ChatPromptTemplate.from_template("""Answer the user's question based on the provided context:
<context>
{context}
</context>
Question: {input}""")
        document_chain = create_stuff_documents_chain(llm, prompt)
        self.retrieval_chain = create_retrieval_chain(retriever, document_chain)
        
        return {"status": "success", "message": "Chatbot backend ready."}

    def get_response(self, user_query):
        """
        Processes a user query and returns a chatbot response.

        Args:
            user_query (str): The user's question.

        Returns:
            dict: A dictionary with the status and the response text.
        """
        if not self.retrieval_chain:
            return {"status": "error", "message": "Chatbot is not initialized. Please load a PDF first."}
        
        try:
            response = self.retrieval_chain.invoke({"input": user_query})
            chatbot_response = response['answer']
            
            # Formatting logic from the original code
            chatbot_response = re.sub(r'\n{2,}', '\n', chatbot_response)
            chatbot_response = re.sub(r'(\d+\.)\s', r'\n\1 ', chatbot_response)
            chatbot_response = re.sub(r'([-*])\s', r'\n\1 ', chatbot_response)
            
            return {"status": "success", "message": chatbot_response}
        except Exception as e:
            return {"status": "error", "message": f"An error occurred: {e}"}

# Example usage of the class
if __name__ == "__main__":
    chatbot = RAGChatbot()
    
    pdf_file_path = "/home/gururaj/Projects/LangChain/data/What_is_a_P_Value_Anyway_Vickers_Andrew.pdf"  # Replace with the path to your PDF
    
    # Initialize the chatbot with the PDF
    init_status = chatbot.initialize_retrieval_chain(pdf_file_path)
    print(init_status['message'])
    
    if init_status['status'] == 'success':
        while True:
            user_input = input("You: ")
            if user_input.lower() in ["exit", "quit"]:
                break
            
            response = chatbot.get_response(user_input)
            if response['status'] == 'success':
                print(f"Chatbot: {response['message']}")
            else:
                print(f"Error: {response['message']}")

Loading existing embeddings...


/tmp/ipykernel_4471/117064951.py:56: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text")
/tmp/ipykernel_4471/117064951.py:57: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vector_store = Chroma(persist_directory=persist_directory, embedding_function=embeddings)
/tmp/ipykernel_4471/117064951.py:72: LangChainDeprecationWarning: The class `O

Chatbot backend ready.
Chatbot: Please provide the question you would like me to answer based on the context.
Chatbot: According to the context, the Bill Gates joke is about the difference between two average salaries – Owen and NWDIA. The joke illustrates that the average salary changes depending on the situation.
